# Kahneman Framing RCT × TRIBE v2 (text only)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/akifnu/DSprojects/blob/main/tribev2/notebooks/Framing_RCT_NoSetup.ipynb)

**318 text pairs** (636 texts) · **63,600** RCT trial assignments

1. Runtime → **A100 GPU**
2. Accept [LLaMA 3.2 license](https://huggingface.co/meta-llama/Llama-3.2-3B)
3. **Run all** twice if runtime restarts after install
4. Paste Hugging Face read token when prompted (or Colab secret `HF_TOKEN`)

In [ ]:
NOTEBOOK_VERSION = 'text-rct-2026-06-17'
print(NOTEBOOK_VERSION, '| text-only | 318 pairs')

In [ ]:
import subprocess, sys, urllib.request
from pathlib import Path

MARKER = Path('/content/.tribev2_colab_v3')
REQ_URL = 'https://raw.githubusercontent.com/akifnu/DSprojects/main/tribev2/requirements-colab.txt'

if not MARKER.exists():
    req = Path('/content/requirements-colab.txt')
    req.write_bytes(urllib.request.urlopen(REQ_URL).read())
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'numpy', 'transformers', 'torch', 'torchvision'], check=False)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req)])
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
        'tribev2 @ git+https://github.com/facebookresearch/TRIBEv2.git',
    ])
    MARKER.write_text('ok')
    print('Pinned torch 2.6 + transformers 4.47.1. Restarting — Run all again.')
    import IPython
    IPython.get_ipython().kernel.do_shutdown(restart=True)
else:
    import torch
    if not hasattr(torch, 'float8_e8m0fnu'):
        torch.float8_e8m0fnu = torch.uint8
    if not hasattr(torch, 'float8_e4m3fn'):
        torch.float8_e4m3fn = torch.uint8
    import numpy as np
    import transformers
    from tribev2 import TribeModel
    from transformers.models.llama.modeling_llama import LlamaModel
    print('torch', torch.__version__, '| transformers', transformers.__version__, '| numpy', np.__version__)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
import os
from getpass import getpass
from huggingface_hub import login

if not os.environ.get('HF_TOKEN'):
    try:
        from google.colab import userdata
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except Exception:
        os.environ['HF_TOKEN'] = getpass('HuggingFace read token: ')

os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '600'
os.environ['HF_HUB_HTTP_TIMEOUT'] = '600'
login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
print('HF login OK')

In [ ]:
import json, urllib.request

BASE = 'https://raw.githubusercontent.com/akifnu/DSprojects/main/tribev2/data/framing_rct'
SCENARIOS = json.loads(urllib.request.urlopen(f'{BASE}/scenarios.json').read().decode())['scenarios']
PROTOCOL = json.loads(urllib.request.urlopen(f'{BASE}/protocol.json').read().decode())
print('scenario pairs:', len(SCENARIOS))
print('unique texts:', len(SCENARIOS) * 2)
print('trial assignments:', PROTOCOL['n_trials'])

In [ ]:
MAX_SCENARIOS = 12  # raise toward 318 for full massive RCT
RUN = SCENARIOS[:MAX_SCENARIOS]
print(f'Will run {len(RUN)} text pairs')

In [ ]:
import torch
from tribev2 import TribeModel

model = TribeModel.from_pretrained('facebook/tribev2', cache_folder='/content/tribe_cache', device='cuda')
print('TRIBE v2 ready')

In [ ]:
import os, tempfile
import numpy as np

def predict_text(text: str) -> np.ndarray:
    tmp = tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False, encoding='utf-8')
    try:
        tmp.write(text.strip())
        tmp.flush()
        os.fsync(tmp.fileno())
        tmp.close()
        events = model.get_events_dataframe(text_path=tmp.name)
        preds, _ = model.predict(events=events, verbose=False)
        return np.asarray(preds)
    finally:
        if os.path.exists(tmp.name):
            os.unlink(tmp.name)

results = []
for s in RUN:
    row = {'id': s['scenario_id'], 'domain': s['domain']}
    for frame, key in [('gain', 'gain_frame'), ('loss', 'loss_frame')]:
        preds = predict_text(s[key])
        row[f'{frame}_mean_abs'] = float(np.mean(np.abs(preds)))
    row['loss_minus_gain'] = row['loss_mean_abs'] - row['gain_mean_abs']
    results.append(row)
    print(s['scenario_id'], f"{row['loss_minus_gain']:+.4f}")

In [ ]:
import pandas as pd
from scipy import stats

df = pd.DataFrame(results)
display(df)
diff = df['loss_mean_abs'].values - df['gain_mean_abs'].values
_, p = stats.ttest_rel(df['loss_mean_abs'], df['gain_mean_abs'])
print(f"loss>gain: {(diff>0).sum()}/{len(df)}  p={p:.4f}")